# 2. Data Preparation and Cleaning

## Overview

This notebook transforms the harmonized BRFSS dataset into an analysis-ready format suitable for statistical analysis and machine learning. The raw data contains response codes, missing values, and categorical variables that require systematic cleaning and recoding.

### Research Importance of Data Preparation

Raw BRFSS survey responses contain coded values, inconsistent formats, and missing responses that can introduce bias and reduce model reliability. Systematic cleaning and recoding ensure that variables are comparable across survey years and suitable for statistical modeling.

Proper data preparation is essential for producing valid estimates of diabetes risk and for ensuring that longitudinal comparisons reflect true population patterns rather than data inconsistencies.

### Key Objectives

1. **Data Quality Assessment**: Examine distributions, missing values, and data types
2. **Variable Recoding**: Transform multi-level categorical variables into binary indicators
3. **Missing Value Treatment**: Implement systematic imputation strategies
4. **Duplicate Removal**: Identify and remove duplicate survey responses
5. **Final Dataset Creation**: Export cleaned data for downstream analysis

### Workflow Summary

```
Harmonized Data → Recode Variables → Handle Missing Values → Remove Duplicates → Feature Engineering → Export Clean Dataset
```

### Input/Output Files

**Input**: 
- `BRFSS_2015_2024_harmonized.zip` ( Harmonized dataset from Notebook 1_data_download.ipynb )

**Output**:
- `BRFSS_2015_2024_cleaned.zip` ( Analysis-ready dataset with binary encodings )
- `BRFSS_2015_2024_recoded.zip` ( Dataset with human-readable labels )

---

## 2.1 Import Required Libraries

Import necessary Python libraries for data manipulation, visualization, and statistical analysis.

In [1]:
from pathlib import Path
from typing import Union, Optional, Dict, List, Tuple
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer

## 2.2 Configuration and File Paths

Define paths for input/output files and configuration mappings.

**Key File Paths**:
- **BRFSS_HARMONIZED_ZIP**: Output file from Notebook 1_data_download.ipynb (harmonized variable names)
- **BRFSS_HARMONIZED_CLEAN_ZIP_FILE**: Output file with binary/numeric encodings for modeling
- **BRFSS_HARMONIZED_RECODED_ZIP_FILE**: Output file with human-readable text labels
- **RECODE_MAP_FILE**: JSON file defining recoding rules
- **RECODED_CONFIG_FILE**: JSON file with text label mappings

**Note**: The Notebook 1 produces *harmonized* (not yet cleaned) data across years 2015-2024

In [2]:
# Use pathlib for robust path handling
BASE_DIR = Path.cwd().parent 
PROCESSED_DATA_DIR = BASE_DIR / "data_processed"
CONFIG_DIR = BASE_DIR / "config"

BRFSS_HARMONIZED_ZIP_FILE = PROCESSED_DATA_DIR / "BRFSS_2015_2024.zip"
BRFSS_HARMONIZED_CLEAN_ZIP_FILE = PROCESSED_DATA_DIR / "BRFSS_2015_2024_cleaned.zip"
BRFSS_HARMONIZED_RECODED_ZIP_FILE = PROCESSED_DATA_DIR / "BRFSS_2015_2024_recoded.zip"
RECODE_MAP_FILE = CONFIG_DIR / "recode_mappings.json"
RECODED_CONFIG_FILE = CONFIG_DIR / "VALUE_RECODED_TEXT_MAP.json"

# Ensure directories exist where the processed data will be saved
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

## 2.3 Utility Functions

These utility functions enable efficient reading and writing of CSV files within ZIP archives, as well as introspecting unique and missing values

**Functions:**

- **`read_zipped_csv()`**: Reads a CSV file directly from a ZIP archive
  - Leverages pandas' native compression support
  - Accepts additional parameters for `read_csv()` (e.g., dtype, parse_dates)

- **`write_df_to_zipped_csv()`**: Writes a DataFrame to a CSV inside a ZIP archive
  - Creates compressed output in a single operation
  - Customizable CSV name within the archive
  - Supports all standard `to_csv()` parameters

- **`unique_values_in_column()`**: Return a DataFrame with unique values in each column of the input DataFrame.

- **`print_unique_values()`**: Print the unique values in each column of the DataFrame along with their data types and counts.

- **`missing_values_summary()`**: Calculate and return a summary of missing values for each column in the DataFrame.

**Benefits**:
- Reduces disk space by ~60-70%
- Maintains compatibility with pandas operations
- Simplifies file management
- Summarizes unique and missing values

In [3]:
def read_zipped_csv(
    zip_path: Union[str, Path],
    **read_csv_kwargs
) -> pd.DataFrame:
    """
    Read a single-CSV .zip file into a pandas DataFrame.

    Parameters
    ----------
    zip_path : str or Path
        Path to the .zip file (containing exactly one CSV, or a CSV
        with the same name as the zip).
    **read_csv_kwargs :
        Any extra keyword args passed through to pandas.read_csv
        (e.g. sep, dtype, parse_dates).

    Returns
    -------
    pd.DataFrame
    """
    zip_path = Path(zip_path)
    # For a single CSV in the zip, this is enough:
    return pd.read_csv(zip_path, compression="zip", **read_csv_kwargs)

In [4]:
def write_df_to_zipped_csv(
    df: pd.DataFrame,
    zip_path: Union[str, Path],
    csv_name: str | None = None,
    **to_csv_kwargs,
) -> None:
    """
    Write a DataFrame to a CSV stored inside a .zip file.

    Parameters
    ----------
    df : pd.DataFrame
        Data to write.
    zip_path : str or Path
        Path to the .zip file to create, e.g. 'data.zip'.
    csv_name : str, optional
        Name of the CSV file inside the zip, e.g. 'data.csv'.
        If None, uses the zip file stem with '.csv'.
    **to_csv_kwargs :
        Extra keyword args passed to DataFrame.to_csv
        (e.g. index=False, sep=',', encoding='utf-8').
    """
    zip_path = Path(zip_path)
    if csv_name is None:
        csv_name = zip_path.stem + ".csv"

    compression_opts = {
        "method": "zip",
        "archive_name": csv_name,
    }

    df.to_csv(
        zip_path,
        header=True,
        compression=compression_opts,
        **to_csv_kwargs,
    )

In [5]:
def unique_values_in_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a DataFrame with unique values in each column of the input DataFrame.
    Parameters
    ----------
        df : pd.DataFrame
            The input DataFrame to analyze.
        
    Returns
    -------
        pd.DataFrame
            A DataFrame containing unique values for each column, along with their data types and counts.
    """
    
    # 1. Initialize an empty list to store row data
    rows_list = []
    
    for column in df.columns:
        unique_column_values = df[column].unique()
        
        # Sort numeric values for better readability
        if pd.api.types.is_numeric_dtype(df[column]):
            unique_column_values = np.sort(unique_column_values)
            
        column_datatype = df[column].dtype
        num_unique_column_values = len(unique_column_values)

        # 2. Append the dictionary to the list (fast)
        rows_list.append({
            "Column": column,
            "Data Type": column_datatype,
            "Unique Values": unique_column_values,
            "Number of Unique Values": num_unique_column_values
        })
    
    # 3. Create the DataFrame once from the list (efficient)
    unique_df = pd.DataFrame(rows_list)
    
    return unique_df

In [6]:
def print_unique_values(df: pd.DataFrame) -> None:
    """
    Print the unique values in each column of the DataFrame along with their data types and counts.
    Parameters
    ----------
        df : pd.DataFrame
            The input DataFrame to analyze.
    Returns
    -------
        None
    """

    # Get the summary dataframe from your previous function
    unique_df = unique_values_in_column(df)
    
    for row in unique_df.to_dict('records'):
        column_name = row["Column"]
        column_datatype = row["Data Type"]
        unique_column_values = row["Unique Values"]
        
        # Note: Ensure this key matches exactly what you defined in the previous function
        # Previous function used: "Number of Unique Values"
        num_unique_column_values = row["Number of Unique Values"]
        
        print(f"{num_unique_column_values} Unique values in column '{column_name}' of type {column_datatype}: {unique_column_values}")

In [7]:
def missing_values_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate and return a summary of missing values for each column in the DataFrame.
    Parameters
    ----------
    df : pd.DataFrame
        The DataFrame for which to calculate missing value statistics.
    Returns
    -------
    pd.DataFrame
        A DataFrame containing the count and percentage of missing values for each column, sorted by percentage of missing values in descending order.
    """
    missing_summary = pd.DataFrame({
        "Column": df.columns,
        "Missing Count": df.isnull().sum(),
        "Missing Percentage": (df.isnull().mean() * 100).round(2)
    }).sort_values(by="Missing Percentage", ascending=False).reset_index(drop=True)
    
    return missing_summary

## 2.4 Load Harmonized Dataset

Load the harmonized BRFSS dataset created in Notebook 1.

**Expected Data Characteristics**:
- **Rows**: ~4,410,255 survey responses
- **Columns**: 19 harmonized variables
- **Data Types**: Mix of int64 and float64 (categorical codes as numbers)
- **Missing Values**: Present as NaN or special codes (7, 9, 77, 88, 99)

**Quality Checks**:
- Verify row count matches expected range
- Confirm all canonical variables are present
- Log initial dataset dimensions

In [8]:
# Load the BRFSS data
harmonized_df = read_zipped_csv(BRFSS_HARMONIZED_ZIP_FILE)
print(f"Initial data loaded with {harmonized_df.shape[0]} rows. and columns: {harmonized_df.shape[1]}")

Initial data loaded with 4410255 rows. and columns: 19


## 2.5 Initial Data Exploration

### Role of Exploratory Checks

Examining unique values and missing data patterns ensures that special response codes are correctly interpreted and prevents misclassification during modeling. This step safeguards analytical accuracy and supports defensible data transformation decisions.

- Examine unique values for each variable to understand:
    - Response code ranges
    - Special missing value codes (7, 9, 77, 88, 99)
    - Data type consistency
- Data information and description
- Missing values count and percentage for 

**Common BRFSS Response Codes**:
- `7`: "Don't know / Not sure"
- `9`: "Refused"
- `77`: "Don't know / Not sure" (alternate)
- `88`: "None" or "Zero" (context-dependent)
- `99`: "Refused" (alternate)

**Purpose**: Identify which values need recoding or treatment as missing data.

In [9]:
# For each column, print unique values
print("Unique values for each column using the original DataFrame:")
print_unique_values(harmonized_df)

Unique values for each column using the original DataFrame:
10 Unique values in column 'YEAR' of type int64: [2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
7 Unique values in column 'DIABETES' of type float64: [ 1.  2.  3.  4.  7.  9. nan]
14 Unique values in column 'AGE_CATEGORIES' of type float64: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14.]
5 Unique values in column 'SEX' of type float64: [ 1.  2.  7.  9. nan]
10 Unique values in column 'RACE' of type float64: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. nan]
7 Unique values in column 'INCOME' of type float64: [ 1.  2.  3.  4.  5.  9. nan]
5 Unique values in column 'EDUCATION_LEVEL' of type float64: [1. 2. 3. 4. 9.]
10 Unique values in column 'EMPLOYMENT' of type float64: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. nan]
8 Unique values in column 'MARITAL_STATUS' of type float64: [ 1.  2.  3.  4.  5.  6.  9. nan]
3 Unique values in column 'HEALTH_CARE_COVERAGE' of type float64: [1. 2. 9.]
5 Unique values in column 'BMICAT' of type

In [10]:
print(f"Original Data information:\n {harmonized_df.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4410255 entries, 0 to 4410254
Data columns (total 19 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   YEAR                       int64  
 1   DIABETES                   float64
 2   AGE_CATEGORIES             float64
 3   SEX                        float64
 4   RACE                       float64
 5   INCOME                     float64
 6   EDUCATION_LEVEL            float64
 7   EMPLOYMENT                 float64
 8   MARITAL_STATUS             float64
 9   HEALTH_CARE_COVERAGE       float64
 10  BMICAT                     float64
 11  HEART_ATTACK               float64
 12  STROKE                     float64
 13  EXERCISE                   float64
 14  SMOKER                     float64
 15  HEAVY_ALCOHOL_CONSUMPTION  float64
 16  HEALTH_STATUS              float64
 17  POOR_PHYSICAL_HEALTH_DAYS  float64
 18  POOR_MENTAL_HEALTH_DAYS    float64
dtypes: float64(18), int64(1)
memory usage: 639

In [11]:
print("Original Data description:\n")
harmonized_df.describe().T

Original Data description:



,count,mean,std,min,25%,50%,75%,max
YEAR,4410255.0,2019.470309,2.905843,2015.0,2017.0,2019.0,2022.0,2024.0
DIABETES,4410187.0,2.750988,0.744697,1.0,3.0,3.0,3.0,9.0
AGE_CATEGORIES,4410255.0,7.772692,3.620983,1.0,5.0,8.0,11.0,14.0
SEX,3050390.0,1.555347,0.541623,1.0,1.0,2.0,2.0,9.0
RACE,4410162.0,2.157547,2.402617,1.0,1.0,1.0,2.0,9.0
INCOME,4045749.0,4.498346,2.090601,1.0,3.0,5.0,5.0,9.0
EDUCATION_LEVEL,4410255.0,3.019977,1.044030,1.0,2.0,3.0,4.0,9.0
EMPLOYMENT,4387101.0,3.905136,2.882087,1.0,1.0,3.0,7.0,9.0
MARITAL_STATUS,4410112.0,2.365037,1.772194,1.0,1.0,1.0,3.0,9.0
HEALTH_CARE_COVERAGE,4410255.0,4.139525,3.849049,1.0,1.0,1.0,9.0,9.0


In [12]:
def missing_values_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate and return a summary of missing values for each column in the DataFrame.
    Parameters
    ----------
    df : pd.DataFrame
        The DataFrame for which to calculate missing value statistics.
    Returns
    -------
    pd.DataFrame
        A DataFrame containing the count and percentage of missing values for each column, sorted by percentage of missing values in descending order.
    """
    missing_summary = pd.DataFrame({
        "Column": df.columns,
        "Missing Count": df.isnull().sum(),
        "Missing Percentage": (df.isnull().mean() * 100).round(2)
    }).sort_values(by="Missing Percentage", ascending=False).reset_index(drop=True)
    
    return missing_summary

In [13]:
print("Missing values summary for the original DataFrame:")
missing_values_summary(harmonized_df)

Missing values summary for the original DataFrame:


,Column,Missing Count,Missing Percentage
0,SEX,1359865,30.83
1,BMICAT,404512,9.17
2,INCOME,364506,8.26
3,EXERCISE,89479,2.03
4,HEART_ATTACK,43524,0.99
5,EMPLOYMENT,23154,0.53
6,YEAR,0,0.00
7,STROKE,52,0.00
8,POOR_PHYSICAL_HEALTH_DAYS,1,0.00
9,HEALTH_STATUS,0,0.00


## 2.6 Data Recoding

### Analytical Rationale for Recoding

BRFSS variables often contain multiple response categories that are not directly suitable for predictive modeling. Recoding simplifies these responses into analytically meaningful categories while preserving substantive interpretation. This improves model stability and enhances interpretability of diabetes risk factors.

Applying `recode_mappings.json` to transform raw survey codes into simplified analytical categories (0-1 binary or ordinal).

In [14]:
def apply_recode_map(df, map_file=RECODE_MAP_FILE, offset=42):
    """
    Apply recoding and missing value handling using the offset method.
    
    The function applies offset to both orig_values and missing_values,
    then replaces:
    - orig_value + offset -> recoded_value
    - missing_value + offset -> NaN
    
    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe to recode
    map_file : str
        Path to the recoded_mappings.json file
    offset : int, default=42
        The offset value for safe recoding
    
    Returns:
    --------
    pd.DataFrame
        The recoded dataframe
    """
    with open(map_file, 'r') as f:
        recode_map = json.load(f)
    
    for col_name, mapping in recode_map.items():
        if col_name not in df.columns:
            print(f"  Column {col_name} not found in dataframe, skipping...")
            continue
        
        print(f"Recoding {col_name}...")
        
        orig_values = mapping['orig_values']
        recoded_values = mapping['recoded_values']
        missing_values = mapping.get('missing_values', [])
        
        # Step 1: Add offset to all values in the column
        temp_values = df[col_name] + offset
        
        # Step 2: Create offset mapping
        # Map: (orig_value + offset) -> recoded_value
        offset_mapping = {
            orig + offset: recoded 
            for orig, recoded in zip(orig_values, recoded_values)
        }
        
        # Map: (missing_value + offset) -> NaN
        for missing_val in missing_values:
            offset_mapping[missing_val + offset] = np.nan
        
        # Step 3: Apply the offset mapping
        df[col_name] = temp_values.map(offset_mapping)
        
        # Report results
        num_recoded = len(orig_values)
        num_missing = len(missing_values)
        num_unmapped = df[col_name].isna().sum()
        print(f"    Recoded {num_recoded} values, set {num_missing} value types to NaN")
        print(f"  Total NaN values in column: {num_unmapped}")
    
    return df


recoded_df = apply_recode_map(harmonized_df)

Recoding DIABETES...
    Recoded 4 values, set 2 value types to NaN
  Total NaN values in column: 9064
Recoding AGE_CATEGORIES...
    Recoded 13 values, set 1 value types to NaN
  Total NaN values in column: 76326
Recoding SEX...
    Recoded 2 values, set 2 value types to NaN
  Total NaN values in column: 1362770
Recoding RACE...
    Recoded 7 values, set 1 value types to NaN
  Total NaN values in column: 186412
Recoding INCOME...
    Recoded 5 values, set 1 value types to NaN
  Total NaN values in column: 836353
Recoding EDUCATION_LEVEL...
    Recoded 4 values, set 1 value types to NaN
  Total NaN values in column: 20184
Recoding EMPLOYMENT...
    Recoded 8 values, set 1 value types to NaN
  Total NaN values in column: 65666
Recoding MARITAL_STATUS...
    Recoded 6 values, set 1 value types to NaN
  Total NaN values in column: 38391
Recoding HEALTH_CARE_COVERAGE...
    Recoded 2 values, set 1 value types to NaN
  Total NaN values in column: 1695764
Recoding BMICAT...
    Recoded 4 val

In [15]:
print("Unique values after recoding:")
print_unique_values(recoded_df)

Unique values after recoding:
10 Unique values in column 'YEAR' of type int64: [2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
5 Unique values in column 'DIABETES' of type float64: [ 0.  1.  2.  3. nan]
14 Unique values in column 'AGE_CATEGORIES' of type float64: [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. nan]
3 Unique values in column 'SEX' of type float64: [ 0.  1. nan]
8 Unique values in column 'RACE' of type float64: [ 0.  1.  2.  3.  4.  5.  7. nan]
6 Unique values in column 'INCOME' of type float64: [ 0.  1.  2.  3.  4. nan]
5 Unique values in column 'EDUCATION_LEVEL' of type float64: [ 0.  1.  2.  3. nan]
9 Unique values in column 'EMPLOYMENT' of type float64: [ 0.  1.  2.  3.  4.  5.  6.  7. nan]
7 Unique values in column 'MARITAL_STATUS' of type float64: [ 0.  1.  2.  3.  4.  5. nan]
3 Unique values in column 'HEALTH_CARE_COVERAGE' of type float64: [ 0.  1. nan]
5 Unique values in column 'BMICAT' of type float64: [ 0.  1.  2.  3. nan]
3 Unique values in column 'H

In [16]:
print(f"Data information:\n {recoded_df.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4410255 entries, 0 to 4410254
Data columns (total 19 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   YEAR                       int64  
 1   DIABETES                   float64
 2   AGE_CATEGORIES             float64
 3   SEX                        float64
 4   RACE                       float64
 5   INCOME                     float64
 6   EDUCATION_LEVEL            float64
 7   EMPLOYMENT                 float64
 8   MARITAL_STATUS             float64
 9   HEALTH_CARE_COVERAGE       float64
 10  BMICAT                     float64
 11  HEART_ATTACK               float64
 12  STROKE                     float64
 13  EXERCISE                   float64
 14  SMOKER                     float64
 15  HEAVY_ALCOHOL_CONSUMPTION  float64
 16  HEALTH_STATUS              float64
 17  POOR_PHYSICAL_HEALTH_DAYS  float64
 18  POOR_MENTAL_HEALTH_DAYS    float64
dtypes: float64(18), int64(1)
memory usage: 639

In [17]:
print("Data description:\n")
recoded_df.describe().T

Data description:



,count,mean,std,min,25%,50%,75%,max
YEAR,4410255.0,2019.470309,2.905843,2015.0,2017.0,2019.0,2022.0,2024.0
DIABETES,4401191.0,0.215606,0.558711,0.0,0.0,0.0,0.0,3.0
AGE_CATEGORIES,4333929.0,6.663021,3.556324,0.0,4.0,7.0,10.0,12.0
SEX,3047485.0,0.451217,0.497615,0.0,0.0,0.0,1.0,1.0
RACE,4223843.0,0.899002,2.097827,0.0,0.0,0.0,0.0,7.0
INCOME,3573902.0,2.904012,1.385254,0.0,2.0,4.0,4.0,4.0
EDUCATION_LEVEL,4390071.0,1.992483,0.964281,0.0,1.0,2.0,3.0,3.0
EMPLOYMENT,4344589.0,5.401515,1.817153,0.0,5.0,6.0,7.0,7.0
MARITAL_STATUS,4371864.0,3.383812,1.933014,0.0,2.0,5.0,5.0,5.0
HEALTH_CARE_COVERAGE,2714491.0,0.896852,0.304152,0.0,1.0,1.0,1.0,1.0


In [18]:
print("Missing values summary for the recoded DataFrame:")
missing_values_summary(recoded_df)

Missing values summary for the recoded DataFrame:


,Column,Missing Count,Missing Percentage
0,HEALTH_CARE_COVERAGE,1695764,38.45
1,SEX,1362770,30.90
2,INCOME,836353,18.96
3,BMICAT,404512,9.17
4,HEAVY_ALCOHOL_CONSUMPTION,334848,7.59
5,SMOKER,228902,5.19
6,RACE,186412,4.23
7,EXERCISE,100960,2.29
8,POOR_PHYSICAL_HEALTH_DAYS,99659,2.26
9,POOR_MENTAL_HEALTH_DAYS,79535,1.80


## 2.7 Data Cleaning

- Drop rows which does not have data for the target variable `DIABETES`
- Drop duplicate rows
- Missing responses in survey data can introduce bias if not handled appropriately. A hybrid imputation strategy is used to balance statistical rigor and interpretability:
    - Explicit "Unknown" categories preserve information where nonresponse may be meaningful.
    - Mode imputation stabilizes low-missing categorical variables.
    - Iterative imputation (MICE) leverages relationships among variables to estimate plausible values.
  
  This approach minimizes bias while maintaining data integrity for modeling.

In [19]:
# Drop rows where DIABETES is value is missing
print("Dropping rows with missing DIABETES values.")
initial_row_count = recoded_df.shape[0]
recoded_df.dropna(subset=['DIABETES'], inplace=True)
print(f"Data after dropping missing DIABETES has {recoded_df.shape[0]} rows (dropped {initial_row_count - recoded_df.shape[0]} rows).")

Dropping rows with missing DIABETES values.
Data after dropping missing DIABETES has 4401191 rows (dropped 9064 rows).


In [20]:
# Drop duplicate rows
print("Dropping duplicate rows from the dataset.")
initial_row_count = recoded_df.shape[0]
recoded_df.drop_duplicates(inplace=True)
print(f"Data after dropping duplicates has {recoded_df.shape[0]} rows (dropped {initial_row_count - recoded_df.shape[0]} duplicates).")

Dropping duplicate rows from the dataset.
Data after dropping duplicates has 3101950 rows (dropped 1299241 duplicates).


In [21]:
print("Data description after dropping missing DIABETES values and duplicate rows:\n")
recoded_df.describe().T

Data description after dropping missing DIABETES values and duplicate rows:



,count,mean,std,min,25%,50%,75%,max
YEAR,3101950.0,2019.469482,2.900146,2015.0,2017.0,2019.0,2022.0,2024.0
DIABETES,3101950.0,0.281001,0.629995,0.0,0.0,0.0,0.0,3.0
AGE_CATEGORIES,3039177.0,6.645169,3.594335,0.0,4.0,7.0,10.0,12.0
SEX,2172537.0,0.440492,0.496446,0.0,0.0,0.0,1.0,1.0
RACE,2931246.0,1.170616,2.329323,0.0,0.0,0.0,1.0,7.0
INCOME,2450147.0,2.550537,1.447042,0.0,1.0,3.0,4.0,4.0
EDUCATION_LEVEL,3083955.0,1.815675,0.975382,0.0,1.0,2.0,3.0,3.0
EMPLOYMENT,3044645.0,5.129752,1.968014,0.0,5.0,5.0,7.0,7.0
MARITAL_STATUS,3067193.0,3.093624,1.954681,0.0,2.0,3.0,5.0,5.0
HEALTH_CARE_COVERAGE,1893344.0,0.857731,0.349326,0.0,1.0,1.0,1.0,1.0


In [22]:
print("Missing values summary after dropping missing DIABETES values and duplicate rows:")
missing_values_summary(recoded_df)

Missing values summary after dropping missing DIABETES values and duplicate rows:


,Column,Missing Count,Missing Percentage
0,HEALTH_CARE_COVERAGE,1208606,38.96
1,SEX,929413,29.96
2,INCOME,651803,21.01
3,BMICAT,331891,10.70
4,HEAVY_ALCOHOL_CONSUMPTION,294826,9.50
5,SMOKER,195277,6.30
6,RACE,170704,5.50
7,POOR_PHYSICAL_HEALTH_DAYS,97085,3.13
8,EXERCISE,90096,2.90
9,POOR_MENTAL_HEALTH_DAYS,77519,2.50


In [23]:
def impute_hybrid_strategy(df: pd.DataFrame, random_state: int = 42) -> pd.DataFrame:
    """
    Impute missing values in a DataFrame using a hybrid strategy.
    The strategy is based on the type of variable and the percentage of missingness:
    - Explicit Unknowns: For high missingness (>5-10%) or sensitive behavioral questions, fill with a specific value (e.g., 99) to indicate "Unknown".
    - Mode Imputation: For low missingness (<5%) and nominal/binary variables, fill with the most frequent value.
    - MICE Imputation: For ordinal/continuous variables where relationships matter, use Iterative Imputer (MICE) to predict missing values based on other features.
    
    Care is taken to avoid imputing the target variable to preserve outcome validity.

    Parameters
    ----------
        df : pd.DataFrame
            The input DataFrame to impute.
        random_state : int, optional
            Random seed for reproducibility (default is 42).
        
    Returns
    -------
        pd.DataFrame
            A DataFrame with missing values imputed using the hybrid strategy.
    """
    df_clean = df.copy(deep=True)
    
    # --- GROUP 1: EXPLICIT UNKNOWN ---
    # High missingness (>5-10%) OR Sensitive Behavioral Questions (Refusal = Signal)
    # Action: Fill with 99 (Make sure 99 is not a valid value in your data!)
    cols_explicit_unknown = [
        'HEALTH_CARE_COVERAGE', # 39% missing
        'SEX',                  # 30% missing
        'INCOME',               # 21% missing
        'BMICAT',               # 10% missing
        'HEAVY_ALCOHOL_CONSUMPTION', # 9.5% missing (Behavioral/Sensitive)
        'EMPLOYMENT'                 # 1.85% missing (Key SES variable, nominal)
    ]
    
    # --- GROUP 2: MODE IMPUTATION ---
    # Low missingness (<5%) AND Nominal/Binary (Yes/No, Categories)
    # Action: Fill with most frequent value
    cols_nominal_mode = [
        'RACE',                 # 5.5% missing
        'MARITAL_STATUS',       # 1.1% missing
        'HEART_ATTACK',         # 1.3% missing
        'STROKE'               # 0.4% missing
    ]
    
    # --- GROUP 3: MICE IMPUTATION ---
    # Ordinal/Continuous variables where relationships matter
    # Action: Iterative Imputer
    cols_ordinal_mice = [
        'AGE_CATEGORIES', 
        'EDUCATION_LEVEL', 
        'HEALTH_STATUS',
        'POOR_PHYSICAL_HEALTH_DAYS', 
        'POOR_MENTAL_HEALTH_DAYS', 
        'SMOKER', 
        'EXERCISE'
    ]
    
    print("Starting Hybrid Imputation...")

    # 1. Apply Explicit Unknowns
    for col in cols_explicit_unknown:
        if col in df_clean.columns:
            # Fill with 99 (Unknown). Ensure your model treats 99 as a category, not a number!
            df_clean[col] = df_clean[col].fillna(99)
            print(f"Filled missing {col} with 99 (Explicit Unknown)")

    # 2. Apply Mode Imputation
    valid_nominal = [c for c in cols_nominal_mode if c in df_clean.columns]
    if valid_nominal:
        imputer_mode = SimpleImputer(strategy='most_frequent')
        df_clean[valid_nominal] = imputer_mode.fit_transform(df_clean[valid_nominal])
        print(f"Imputed {len(valid_nominal)} columns using Mode (Most Frequent)")

    # 3. Apply MICE Imputation
    valid_ordinal = [c for c in cols_ordinal_mice if c in df_clean.columns]
    if valid_ordinal:
        imputer_mice = IterativeImputer(random_state=random_state)
        # Fit MICE only on the ordinal columns
        df_clean[valid_ordinal] = imputer_mice.fit_transform(df_clean[valid_ordinal])
        
        # Round MICE outputs to nearest integer (e.g., 2.3 -> 2)
        df_clean[valid_ordinal] = df_clean[valid_ordinal].round().astype(int)
        print(f"Imputed {len(valid_ordinal)} columns using MICE")

    return df_clean

In [24]:
imputed_df = impute_hybrid_strategy(recoded_df)

Starting Hybrid Imputation...
Filled missing HEALTH_CARE_COVERAGE with 99 (Explicit Unknown)
Filled missing SEX with 99 (Explicit Unknown)
Filled missing INCOME with 99 (Explicit Unknown)
Filled missing BMICAT with 99 (Explicit Unknown)
Filled missing HEAVY_ALCOHOL_CONSUMPTION with 99 (Explicit Unknown)
Filled missing EMPLOYMENT with 99 (Explicit Unknown)
Imputed 4 columns using Mode (Most Frequent)
Imputed 7 columns using MICE


In [25]:
print("Missing values summary after imputation:")
missing_values_summary(imputed_df)

Missing values summary after imputation:


,Column,Missing Count,Missing Percentage
0,YEAR,0,0.0
1,BMICAT,0,0.0
2,POOR_PHYSICAL_HEALTH_DAYS,0,0.0
3,HEALTH_STATUS,0,0.0
4,HEAVY_ALCOHOL_CONSUMPTION,0,0.0
5,SMOKER,0,0.0
6,EXERCISE,0,0.0
7,STROKE,0,0.0
8,HEART_ATTACK,0,0.0
9,HEALTH_CARE_COVERAGE,0,0.0


No missing values

In [26]:
# For each column, print unique values
print("Print unique values after imputing the data")
print_unique_values(imputed_df)

Print unique values after imputing the data
10 Unique values in column 'YEAR' of type int64: [2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
4 Unique values in column 'DIABETES' of type float64: [0. 1. 2. 3.]
13 Unique values in column 'AGE_CATEGORIES' of type int32: [ 0  1  2  3  4  5  6  7  8  9 10 11 12]
3 Unique values in column 'SEX' of type float64: [ 0.  1. 99.]
7 Unique values in column 'RACE' of type float64: [0. 1. 2. 3. 4. 5. 7.]
6 Unique values in column 'INCOME' of type float64: [ 0.  1.  2.  3.  4. 99.]
4 Unique values in column 'EDUCATION_LEVEL' of type int32: [0 1 2 3]
9 Unique values in column 'EMPLOYMENT' of type float64: [ 0.  1.  2.  3.  4.  5.  6.  7. 99.]
6 Unique values in column 'MARITAL_STATUS' of type float64: [0. 1. 2. 3. 4. 5.]
3 Unique values in column 'HEALTH_CARE_COVERAGE' of type float64: [ 0.  1. 99.]
5 Unique values in column 'BMICAT' of type float64: [ 0.  1.  2.  3. 99.]
2 Unique values in column 'HEART_ATTACK' of type float64: [0. 1.]
2 Unique v

## 2.8 Feature Engineering

Derived variables, such as cumulative risk indicators, capture the combined burden of behavioral and health risk factors. These features support analysis of compounding risk effects and enable more comprehensive modeling of diabetes vulnerability.

In [27]:
# Move prediabetes having a recoded value of 2 to a separate column and replace the original value with 0 (no diabetes)
imputed_df['PRE_DIABETES'] = imputed_df['DIABETES'].apply(lambda x: 1 if x == 3 else 0)
imputed_df['DIABETES'] = imputed_df['DIABETES'].apply(lambda x: 0 if x == 3 else x)

# Move gestational diabetes having a recoded value of 1 to a separate column and replace the original value with 0 (no diabetes)
imputed_df['GEST_DIABETES'] = imputed_df['DIABETES'].apply(lambda x: 1 if x == 2 else 0)
imputed_df['DIABETES'] = imputed_df['DIABETES'].apply(lambda x: 0 if x == 2 else x)

# 1. Generation Flag (RQ1)
# Assuming 'AGE_CATEGORIES' 0=18-24, 1=25-29
imputed_df['IS_YOUNG_ADULT'] = imputed_df['AGE_CATEGORIES'].isin([0, 1]).astype(int)

# 2. Pandemic Flag 0 if pre-pandemic (2015-2019), 1 if pandemic (2020-2021), 2 if post-pandemic (2022-2024)
imputed_df['PANDEMIC_FLAG'] = imputed_df['YEAR'].apply(lambda x: 0 if x <= 2019 else (1 if x <= 2021 else 2))

# 3. Syndemic Risk Score (RQ4)
imputed_df['RISK_OBESITY'] = (imputed_df['BMICAT'] == 3).astype(int)
imputed_df['RISK_INACTIVE'] = (imputed_df['EXERCISE'] == 0).astype(int)
imputed_df['RISK_SMOKING'] = imputed_df['SMOKER'].isin([0, 1]).astype(int)

imputed_df['TOTAL_RISK_FACTORS'] = (imputed_df['RISK_OBESITY'] + imputed_df['RISK_INACTIVE'] + imputed_df['RISK_SMOKING'])

In [28]:
# For each column, print unique values
print("Print unique values after Feature Engineering")
print_unique_values(imputed_df)

Print unique values after Feature Engineering
10 Unique values in column 'YEAR' of type int64: [2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
2 Unique values in column 'DIABETES' of type float64: [0. 1.]
13 Unique values in column 'AGE_CATEGORIES' of type int32: [ 0  1  2  3  4  5  6  7  8  9 10 11 12]
3 Unique values in column 'SEX' of type float64: [ 0.  1. 99.]
7 Unique values in column 'RACE' of type float64: [0. 1. 2. 3. 4. 5. 7.]
6 Unique values in column 'INCOME' of type float64: [ 0.  1.  2.  3.  4. 99.]
4 Unique values in column 'EDUCATION_LEVEL' of type int32: [0 1 2 3]
9 Unique values in column 'EMPLOYMENT' of type float64: [ 0.  1.  2.  3.  4.  5.  6.  7. 99.]
6 Unique values in column 'MARITAL_STATUS' of type float64: [0. 1. 2. 3. 4. 5.]
3 Unique values in column 'HEALTH_CARE_COVERAGE' of type float64: [ 0.  1. 99.]
5 Unique values in column 'BMICAT' of type float64: [ 0.  1.  2.  3. 99.]
2 Unique values in column 'HEART_ATTACK' of type float64: [0. 1.]
2 Unique value

In [29]:
print(f"Data information:\n {imputed_df.info()}")

<class 'pandas.core.frame.DataFrame'>
Index: 3101950 entries, 0 to 4410254
Data columns (total 27 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   YEAR                       int64  
 1   DIABETES                   float64
 2   AGE_CATEGORIES             int32  
 3   SEX                        float64
 4   RACE                       float64
 5   INCOME                     float64
 6   EDUCATION_LEVEL            int32  
 7   EMPLOYMENT                 float64
 8   MARITAL_STATUS             float64
 9   HEALTH_CARE_COVERAGE       float64
 10  BMICAT                     float64
 11  HEART_ATTACK               float64
 12  STROKE                     float64
 13  EXERCISE                   int32  
 14  SMOKER                     int32  
 15  HEAVY_ALCOHOL_CONSUMPTION  float64
 16  HEALTH_STATUS              int32  
 17  POOR_PHYSICAL_HEALTH_DAYS  int32  
 18  POOR_MENTAL_HEALTH_DAYS    int32  
 19  PRE_DIABETES               int64  
 20  GEST_DI

In [30]:
print("Data description:\n")
imputed_df.describe().T

Data description:



,count,mean,std,min,25%,50%,75%,max
YEAR,3101950.0,2019.469482,2.900146,2015.0,2017.0,2019.0,2022.0,2024.0
DIABETES,3101950.0,0.172628,0.377925,0.0,0.0,0.0,0.0,1.0
AGE_CATEGORIES,3101950.0,6.646951,3.560413,0.0,4.0,7.0,9.0,12.0
SEX,3101950.0,29.971106,45.151289,0.0,0.0,1.0,99.0,99.0
RACE,3101950.0,1.106195,2.280005,0.0,0.0,0.0,1.0,7.0
INCOME,3101950.0,22.817160,39.314416,0.0,2.0,4.0,4.0,99.0
EDUCATION_LEVEL,3101950.0,1.816179,0.972833,0.0,1.0,2.0,3.0,3.0
EMPLOYMENT,3101950.0,6.863898,12.789802,0.0,5.0,5.0,7.0,99.0
MARITAL_STATUS,3101950.0,3.114985,1.954029,0.0,2.0,3.0,5.0,5.0
HEALTH_CARE_COVERAGE,3101950.0,39.096689,47.861419,0.0,1.0,1.0,99.0,99.0


In [31]:
imputed_df.info("First few rows of the dataset:")
imputed_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 3101950 entries, 0 to 4410254
Data columns (total 27 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   YEAR                       int64  
 1   DIABETES                   float64
 2   AGE_CATEGORIES             int32  
 3   SEX                        float64
 4   RACE                       float64
 5   INCOME                     float64
 6   EDUCATION_LEVEL            int32  
 7   EMPLOYMENT                 float64
 8   MARITAL_STATUS             float64
 9   HEALTH_CARE_COVERAGE       float64
 10  BMICAT                     float64
 11  HEART_ATTACK               float64
 12  STROKE                     float64
 13  EXERCISE                   int32  
 14  SMOKER                     int32  
 15  HEAVY_ALCOHOL_CONSUMPTION  float64
 16  HEALTH_STATUS              int32  
 17  POOR_PHYSICAL_HEALTH_DAYS  int32  
 18  POOR_MENTAL_HEALTH_DAYS    int32  
 19  PRE_DIABETES               int64  
 20  GEST_DI

,YEAR,DIABETES,AGE_CATEGORIES,SEX,RACE,INCOME,EDUCATION_LEVEL,EMPLOYMENT,MARITAL_STATUS,HEALTH_CARE_COVERAGE,...,POOR_PHYSICAL_HEALTH_DAYS,POOR_MENTAL_HEALTH_DAYS,PRE_DIABETES,GEST_DIABETES,IS_YOUNG_ADULT,PANDEMIC_FLAG,RISK_OBESITY,RISK_INACTIVE,RISK_SMOKING,TOTAL_RISK_FACTORS
0,2015,0.0,8,0.0,0.0,1.0,1,2.0,5.0,1.0,...,2,2,0,0,0,0,1,1,1,3
1,2015,0.0,6,0.0,0.0,0.0,3,0.0,2.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,2015,0.0,10,0.0,0.0,99.0,1,5.0,2.0,99.0,...,2,0,0,0,0,0,0,1,1,2
3,2015,0.0,8,0.0,0.0,4.0,1,2.0,5.0,1.0,...,2,2,0,0,0,0,0,1,1,2
4,2015,0.0,8,0.0,0.0,99.0,2,2.0,5.0,1.0,...,2,0,0,0,0,0,0,1,1,2


In [32]:
print(f"The BRFSS 2015-2024 dataset has been successfully prepared and is ready for analysis and had {imputed_df.shape[0]} rows and {imputed_df.shape[1]} columns.")

The BRFSS 2015-2024 dataset has been successfully prepared and is ready for analysis and had 3101950 rows and 27 columns.


In [33]:
print("Write the cleaned dataset to csv")
write_df_to_zipped_csv(imputed_df, BRFSS_HARMONIZED_CLEAN_ZIP_FILE, index=False)

Write the cleaned dataset to csv


## 2.9 Data Labelling

Generating a labeled dataset enhances interpretability and supports exploratory analysis, reporting, and stakeholder communication without requiring reference to technical codebooks.

Apply the labels based on the `VALUE_RECODED_TEXT_MAP.json` so that the generated csv file is human readable without requiring a reference to the codebooks

In [34]:
def decode_enum_values(df: pd.DataFrame, 
                       mapping_path: str | Path = RECODED_CONFIG_FILE) -> pd.DataFrame:
    """
    Loads a JSON mapping and replaces encoded values in the DataFrame 
    with their descriptive text labels.
    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame with encoded values.
    mapping_path : str or Path
        Path to the JSON file containing the mapping of encoded values to descriptive labels.
    Returns
    -------
    pd.DataFrame
        A DataFrame with encoded values replaced by their descriptive text labels.
    """
    df_decoded = df.copy()
    mapping_path = Path(mapping_path)
    
    with mapping_path.open("r") as f:
        value_map = json.load(f)

    for col, mapping in value_map.items():
        if col not in df_decoded.columns:
            continue

        # Build the conversion dictionary (handling int vs string keys)
        col_map = {}
        for k, v in mapping.items():
            try:
                col_map[int(k)] = v
            except (ValueError, TypeError):
                col_map[str(k)] = v

        # Apply mapping based on data type
        if pd.api.types.is_numeric_dtype(df_decoded[col]):
            df_decoded[col] = (
                df_decoded[col]
                .dropna()
                .astype(int)
                .map(col_map)
                .reindex(df_decoded.index)
            )
        else:
            df_decoded[col] = df_decoded[col].map(col_map)

    return df_decoded

print("Decoding the enum values and generating a decoded dataframe")
decoded_imputed_df = decode_enum_values(imputed_df, RECODED_CONFIG_FILE)
print(f"Writing the decoded dataframe to csv zip file {BRFSS_HARMONIZED_RECODED_ZIP_FILE}")
write_df_to_zipped_csv(decoded_imputed_df, BRFSS_HARMONIZED_RECODED_ZIP_FILE, index=False)

decoded_imputed_df.head()

Decoding the enum values and generating a decoded dataframe
Writing the decoded dataframe to csv zip file c:\github\brfss-diabetes-trends\data_processed\BRFSS_2015_2024_recoded.zip


,YEAR,DIABETES,AGE_CATEGORIES,SEX,RACE,INCOME,EDUCATION_LEVEL,EMPLOYMENT,MARITAL_STATUS,HEALTH_CARE_COVERAGE,...,POOR_PHYSICAL_HEALTH_DAYS,POOR_MENTAL_HEALTH_DAYS,PRE_DIABETES,GEST_DIABETES,IS_YOUNG_ADULT,PANDEMIC_FLAG,RISK_OBESITY,RISK_INACTIVE,RISK_SMOKING,TOTAL_RISK_FACTORS
0,2015,No,60-64,Female,White,$15000 - $24999,High School Graduate,Unable to work,Married,Yes,...,14+ days,14+ days,No,No,30+,Pre-Pandemic,Yes,Yes,Yes,3
1,2015,No,50-54,Female,White,< $15000,Graduate,Unemployed > 1 year,Divorced,No,...,Zero days,Zero days,No,No,30+,Pre-Pandemic,No,No,No,0
2,2015,No,70-74,Female,White,Unknown,High School Graduate,Retired,Divorced,Unknown,...,14+ days,Zero days,No,No,30+,Pre-Pandemic,No,Yes,Yes,2
3,2015,No,60-64,Female,White,> $50000,High School Graduate,Unable to work,Married,Yes,...,14+ days,14+ days,No,No,30+,Pre-Pandemic,No,Yes,Yes,2
4,2015,No,60-64,Female,White,Unknown,Undergraduate,Unable to work,Married,Yes,...,14+ days,Zero days,No,No,30+,Pre-Pandemic,No,Yes,Yes,2


In [35]:
# For each column, print unique values
print("Print unique values after decoding the data")
print_unique_values(decoded_imputed_df)

Print unique values after decoding the data
10 Unique values in column 'YEAR' of type int64: [2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
2 Unique values in column 'DIABETES' of type object: ['No' 'Yes']
13 Unique values in column 'AGE_CATEGORIES' of type object: ['60-64' '50-54' '70-74' '80+' '65-69' '75-79' '55-59' '35-39' '45-49'
 '25-29' '30-34' '40-44' '18-24']
3 Unique values in column 'SEX' of type object: ['Female' 'Male' 'Unknown']
7 Unique values in column 'RACE' of type object: ['White' 'Black' 'American Indian' 'Other' 'Hispanic' 'Asian' 'Hawaiian']
6 Unique values in column 'INCOME' of type object: ['$15000 - $24999' '< $15000' 'Unknown' '> $50000' '$35000 - $49999'
 '$25000 - $34999']
4 Unique values in column 'EDUCATION_LEVEL' of type object: ['High School Graduate' 'Graduate' 'Undergraduate' 'Dropout']
9 Unique values in column 'EMPLOYMENT' of type object: ['Unable to work' 'Unemployed > 1 year' 'Retired' 'Self-Employed'
 'Homemaker' 'Unknown' 'Wages' 'Unemployed

In [36]:
print(f"Data information:\n {decoded_imputed_df.info()}")

<class 'pandas.core.frame.DataFrame'>
Index: 3101950 entries, 0 to 4410254
Data columns (total 27 columns):
 #   Column                     Dtype 
---  ------                     ----- 
 0   YEAR                       int64 
 1   DIABETES                   object
 2   AGE_CATEGORIES             object
 3   SEX                        object
 4   RACE                       object
 5   INCOME                     object
 6   EDUCATION_LEVEL            object
 7   EMPLOYMENT                 object
 8   MARITAL_STATUS             object
 9   HEALTH_CARE_COVERAGE       object
 10  BMICAT                     object
 11  HEART_ATTACK               object
 12  STROKE                     object
 13  EXERCISE                   object
 14  SMOKER                     object
 15  HEAVY_ALCOHOL_CONSUMPTION  object
 16  HEALTH_STATUS              object
 17  POOR_PHYSICAL_HEALTH_DAYS  object
 18  POOR_MENTAL_HEALTH_DAYS    object
 19  PRE_DIABETES               object
 20  GEST_DIABETES              ob

In [37]:
print("Data description after decoding:\n")
decoded_imputed_df.describe().T

Data description after decoding:



,count,mean,std,min,25%,50%,75%,max
YEAR,3101950.0,2019.469482,2.900146,2015.0,2017.0,2019.0,2022.0,2024.0
TOTAL_RISK_FACTORS,3101950.0,1.458740,0.775633,0.0,1.0,1.0,2.0,3.0


In [38]:
print("Missing values summary for the decoded DataFrame:")
missing_values_summary(decoded_imputed_df)

Missing values summary for the decoded DataFrame:


,Column,Missing Count,Missing Percentage
0,YEAR,0,0.0
1,SMOKER,0,0.0
2,RISK_SMOKING,0,0.0
3,RISK_INACTIVE,0,0.0
4,RISK_OBESITY,0,0.0
5,PANDEMIC_FLAG,0,0.0
6,IS_YOUNG_ADULT,0,0.0
7,GEST_DIABETES,0,0.0
8,PRE_DIABETES,0,0.0
9,POOR_MENTAL_HEALTH_DAYS,0,0.0


In [39]:
def sample_rows_by_year(df: pd.DataFrame, 
                       year_col: str = "YEAR", 
                       n_per_year: int = 3, 
                       random_state: int | None = None) -> pd.DataFrame:
    """
    Groups the dataframe by year and samples n rows from each group.
    """
    return (
        df
        .groupby(year_col, group_keys=False)
        .apply(lambda g: g.sample(
            n=min(len(g), n_per_year),
            random_state=random_state
        ))
        .reset_index(drop=True)
    )

df_sampled = sample_rows_by_year(decoded_imputed_df)
df_sampled.head(n=30)

,YEAR,DIABETES,AGE_CATEGORIES,SEX,RACE,INCOME,EDUCATION_LEVEL,EMPLOYMENT,MARITAL_STATUS,HEALTH_CARE_COVERAGE,...,POOR_PHYSICAL_HEALTH_DAYS,POOR_MENTAL_HEALTH_DAYS,PRE_DIABETES,GEST_DIABETES,IS_YOUNG_ADULT,PANDEMIC_FLAG,RISK_OBESITY,RISK_INACTIVE,RISK_SMOKING,TOTAL_RISK_FACTORS
0,2015,No,45-49,Female,White,Unknown,Graduate,Wages,Single,Yes,...,Zero days,Zero days,No,No,30+,Pre-Pandemic,No,No,Yes,1
1,2015,Yes,60-64,Male,White,> $50000,Graduate,Wages,Married,Yes,...,14+ days,Zero days,No,No,30+,Pre-Pandemic,Yes,Yes,Yes,3
2,2015,Yes,55-59,Female,White,> $50000,High School Graduate,Wages,Married,Yes,...,Zero days,1-13 days,No,No,30+,Pre-Pandemic,Yes,Yes,Yes,3
3,2016,No,35-39,Female,White,$25000 - $34999,Graduate,Homemaker,Married,Yes,...,Zero days,Zero days,No,No,30+,Pre-Pandemic,No,No,Yes,1
4,2016,No,80+,Male,White,$25000 - $34999,Undergraduate,Retired,Widowed,Unknown,...,14+ days,Zero days,No,No,30+,Pre-Pandemic,No,Yes,Yes,2
5,2016,No,30-34,Male,White,$35000 - $49999,Undergraduate,Wages,Single,Yes,...,1-13 days,1-13 days,No,No,30+,Pre-Pandemic,No,Yes,No,1
6,2017,Yes,75-79,Female,White,$25000 - $34999,Dropout,Homemaker,Married,Unknown,...,1-13 days,Zero days,No,No,30+,Pre-Pandemic,No,Yes,Yes,2
7,2017,No,70-74,Female,White,$25000 - $34999,Undergraduate,Retired,Married,Unknown,...,Zero days,Zero days,No,No,30+,Pre-Pandemic,Yes,Yes,Yes,3
8,2017,No,60-64,Female,White,$25000 - $34999,Graduate,Retired,Divorced,Yes,...,Zero days,Zero days,No,No,30+,Pre-Pandemic,No,No,Yes,1
9,2018,No,40-44,Female,American Indian,$35000 - $49999,High School Graduate,Unable to work,Married,No,...,14+ days,14+ days,No,No,30+,Pre-Pandemic,No,Yes,No,1


## Notebook Outcome

This notebook produces a cleaned and standardized dataset suitable for statistical analysis and machine learning. By addressing missing data, harmonizing variable formats, and engineering meaningful features, the resulting dataset supports reliable modeling and longitudinal analysis of diabetes risk.